# 19 · The told model: how exact the QP has to be, and how it ships

Notebook 18 trained notebook 7's body told each patch's slice QP. Two questions before it can ship:

- **A wrong QP.** The model given the QP off by a few steps: how exact the number has to be. The client reads it off
  the slice header, so it is exact; a client that baked a few graphs and picked the nearest would be off by a few.
- **Biases that are a line in the QP.** Each shift is a 1×1 convolution of the QP with no bias, so a graph baked for
  one QP has the biases `b + w * QP / 51`. The client can compute them every frame from the QP it reads, with the
  blind graph's cost, instead of picking from a few baked graphs.

This notebook trains nothing: it loads notebook 18's told runs and scores them on notebook 0's validation split.

Needs: `18_upscale_qp_models.ipynb` (its runs in `checkpoints/qp_runs/`).

In [1]:
"""Settings."""

import os

import numpy as np
import torch

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, training
from helpers.training import DATA_DIR, DEVICE

WIDTH, BLOCKS, STEM = 24, 2, 3               # notebook 7's body
QP_MAX = patches.QP_MAX                      # the model is told QP / QP_MAX
QP_ERRORS = (-12, -6, -3, 3, 6, 12)          # how far off the told model is told, in QP steps
RUNS = {seed: training.CHECKPOINT_DIR / "qp_runs" / f"told_qp_seed{seed}.pt" for seed in (0, 1)}   # notebook 18's
COLUMNS = patches.columns(("qp",))

manifest = training.load_manifest()
SCALE = manifest["scale"]


def told_model():
    return models.ToldLuma(1, WIDTH, BLOCKS, STEM, scale=SCALE).to(DEVICE)


def biases(model, qp):
    """The three biases a graph baked from told `model` for `qp` has, end to end."""
    plain = model.baked(qp / QP_MAX)
    return torch.cat([plain.body[0].bias, *[block.conv.bias for block in plain.body[2:]]])

## 1. The biases are a line in the QP

`ToldLuma.baked(c)` adds `shift(c)` to three biases, and each shift is a 1×1 convolution of one number with no bias of
its own: `w * c`. So a graph baked for a QP has the biases `b + w * QP / 51`, with `w` 24 numbers for each of the
three. Checked on random weights: a graph baked at any QP has the biases of the line through QP 0 and QP 51, and
computes what the told model computes.

In [2]:
probe = told_model()
with torch.no_grad(), training.full_precision():
    for weight in probe.parameters():
        if weight.requires_grad:
            weight.normal_(0, 0.1)
    at_0, at_max = biases(probe, 0), biases(probe, QP_MAX)
    x = torch.rand(2, 3, 40, 40, device=DEVICE)
    for qp in (7, 23.5, 41):
        off_line = (biases(probe, qp) - (at_0 + (at_max - at_0) * qp / QP_MAX)).abs().max().item()
        computes = (probe(x, torch.full((2, 1, 1, 1), qp / QP_MAX, device=DEVICE))
                    - probe.baked(qp / QP_MAX)(x)).abs().max().item()
        assert off_line < 1e-5, f"baked at QP {qp}, the biases are off the line by {off_line:.1e}"
        assert computes < 1e-5, f"baked at QP {qp}, the graph computes something else ({computes:.1e})"
print(f"a graph baked for a QP: biases b + w * QP / {QP_MAX}, {at_0.numel()} of them, and the told model's output")

a graph baked for a QP: biases b + w * QP / 51, 72 of them, and the told model's output


## 2. A wrong QP

The told model at the seed that did better on the select split, given every patch's QP off by each of `QP_ERRORS`
(kept inside 0 to 51): SSIM-Y and luma dB against the same model told the true QP, by slice QP bin. Every +6 doubles
the quantiser's step.

In [3]:
select_scores = {seed: torch.load(path, map_location="cpu", weights_only=True)["best"]["score"]
                 for seed, path in RUNS.items()}
seed = max(select_scores, key=select_scores.get)
model = told_model()
model.load_state_dict(training.fitted_weights(RUNS[seed]))
model.eval()


def told_off_by(error):
    return lambda lr, condition: model(lr, (condition[:, COLUMNS] + error / QP_MAX).clamp(0, 1))


true = training.val_scores(told_off_by(0), manifest)
wrong = {error: training.val_scores(told_off_by(error), manifest) for error in QP_ERRORS}

qp = np.load(DATA_DIR / manifest["splits"]["val"]["qp"])
edges = manifest["qp_edges"]
print(f"told the QP, seed {seed}; error in QP steps: SSIM-Y / luma dB against the true QP")
print(f"{'QP':>7}{'patches':>9}" + "".join(f"{error:>+16d}" for error in QP_ERRORS))
for low, high in [(0, 99), *zip(edges[:-1], edges[1:])]:
    inside = (qp >= low) & (qp < high)
    cells = [f"{(w['ssim'] - true['ssim'])[inside].mean():+.4f} / {(w['y'] - true['y'])[inside].mean():+.2f}"
             for w in wrong.values()]
    print(f"{'all' if high == 99 else f'{low}-{high}':>7}{inside.sum():9}" + "".join(f"{cell:>16}" for cell in cells))

told the QP, seed 0; error in QP steps: SSIM-Y / luma dB against the true QP
     QP  patches             -12              -6              -3              +3              +6             +12
    all    24286 -0.0062 / -0.06 -0.0019 / +0.02 -0.0005 / +0.02 -0.0005 / -0.06 -0.0021 / -0.17 -0.0070 / -0.47
   0-18     3610 -0.0008 / -0.09 -0.0003 / -0.03 -0.0001 / -0.01 -0.0001 / -0.01 -0.0004 / -0.06 -0.0023 / -0.29
  18-26     3928 -0.0026 / -0.03 -0.0007 / +0.03 -0.0002 / +0.03 -0.0004 / -0.08 -0.0017 / -0.23 -0.0095 / -0.81
  26-30     3730 -0.0052 / -0.02 -0.0018 / +0.05 -0.0006 / +0.05 -0.0009 / -0.12 -0.0041 / -0.33 -0.0135 / -0.80
  30-34     3585 -0.0076 / -0.03 -0.0029 / +0.04 -0.0008 / +0.05 -0.0011 / -0.11 -0.0037 / -0.25 -0.0096 / -0.58
  34-38     3539 -0.0106 / -0.07 -0.0035 / +0.03 -0.0009 / +0.04 -0.0006 / -0.07 -0.0020 / -0.16 -0.0064 / -0.38
  38-42     2846 -0.0118 / -0.12 -0.0033 / -0.01 -0.0009 / +0.01 -0.0001 / -0.03 -0.0011 / -0.07 -0.0045 / -0.18
  42-46     2005 -0

## 3. How much the model leans on the QP

How far the QP moves each layer's biases from QP 0 to 51, against their size at QP 0.

In [4]:
with torch.no_grad():
    at_0, at_max = biases(model, 0), biases(model, QP_MAX)
for layer, start in zip(("stem", "block 1", "block 2"), (0, WIDTH, 2 * WIDTH)):
    moved, size = (at_max - at_0)[start:start + WIDTH].abs(), at_0[start:start + WIDTH].abs()
    print(f"{layer:<8} QP 0 to {QP_MAX} moves a bias {moved.mean():.3f} on average (largest {moved.max():.3f}); "
          f"the biases at QP 0 average {size.mean():.3f}")

stem     QP 0 to 51 moves a bias 0.085 on average (largest 0.505); the biases at QP 0 average 0.183
block 1  QP 0 to 51 moves a bias 0.084 on average (largest 0.289); the biases at QP 0 average 0.026
block 2  QP 0 to 51 moves a bias 0.157 on average (largest 0.399); the biases at QP 0 average 0.126


## 4. What it says

- **The told model ships at the blind graph's cost.** A graph baked for a QP has 72 biases that are a line in the QP,
  `b + w * QP / 51`, checked against the told model's output. The client reads the slice QP off every slice header and
  computes the biases each frame: one graph, nothing to pick between.
- **The QP has to be within a few steps.** Told the QP 3 steps off, the model loses 0.0005 SSIM-Y, a fifth of the
  +0.0027 that telling it gains over the blind model (notebook 18). 6 steps off it loses 0.0019 to 0.0021, most of the
  gain; 12 off it loses 0.0062 to 0.0070 and lands below the blind model. A client that baked a few graphs instead
  would need one every 6 QP steps, at most 3 from the truth, to keep four fifths of the gain.
- **Too high a QP costs more than too low.** Told a QP 12 steps too high, the model loses 0.47 dB of luma; 12 too low,
  0.06 dB. Slice QP 26 to 38 is the most sensitive: 12 steps too high there costs 0.0064 to 0.0135 SSIM-Y.
- **Every layer leans on the QP.** From QP 0 to 51 the biases move 0.08 to 0.16 on average. Block 1's move three times
  their size at QP 0 (0.084 against 0.026).